In [376]:
from datasets import load_dataset
from src.config import cfg
import re
import string
import unicodedata
from collections import Counter

In [352]:
ds = load_dataset(cfg.dataset.path, cfg.dataset.name)

In [353]:
WIKIPEDIA_SEP = [
    "\n\nNotlar",
    "\n\nKaynakça",
]

In [354]:
def strip_wikipedia_templates(text, debug=False):
    # {{...}} içeriğini yakala
    pattern = r"\{\{(.*?)\}\}"

    def repl(match):
        content = match.group(1)

        if debug:
            print("\n--- TEMPLATE FOUND ---")
            print(repr(content[:300]))
            print("--- END TEMPLATE ---\n")

        parts = content.split("|")

        kept = []

        for part in parts:
            if "=" in part:
                key, value = part.split("=", 1)

                key = key.strip().lower()

                # anlamlı alanları koru
                if key in {"alıntı", "metin", "içerik"}:
                    kept.append(value)

            else:
                # template adı (örn: Alıntı kutusu) → ignore
                pass

        return " ".join(kept)

    return re.sub(pattern, repl, text, flags=re.DOTALL)

In [355]:
def strip_wikipedia_sections(text, debug=False):
    for sep in WIKIPEDIA_SEP:
        idx = text.find(sep)

        if idx != -1:
            if debug:
                print("\n--- MATCH FOUND ---")
                print("sep:", repr(sep))
                print(repr(text[idx:idx + 150]))
                print("--- STRIPPING ---")

            text = text[:idx]

    return text

In [367]:
def strip_html(text, debug=False):
    def repl(match):
        if debug:
            print("\n--- HTML TAG FOUND ---")
            print(repr(match.group(0)))
            print("--- END HTML TAG ---\n")

        return " "

    return re.sub(
        r"<[^>]+>",
        repl,
        text,
    )

In [366]:
def strip_wikipedia_tables(text, debug=False):
    result = []
    i = 0
    depth = 0

    while i < len(text):
        if text.startswith("{|", i):
            depth += 1

            if debug:
                print("\n--- TABLE START ---")
                print(repr(text[i:i + 200]))

            i += 2
            continue

        if depth > 0 and text.startswith("|}", i):
            depth -= 1
            i += 2

            if debug:
                print("--- TABLE END ---\n")

            continue

        if depth == 0:
            result.append(text[i])

        i += 1

    return "".join(result)

In [357]:
def clean_text(text):
    text = unicodedata.normalize("NFC", text)

    text = text.lower().replace("i\u0307", "i")

    text = re.sub(r"[\'\"`’‘“”]", "", text)

    # sayı-sayı arasındaki tire → boşluk
    text = re.sub(r"(?<=\d)-(?=\d)", " ", text)

    # harf-harf olmayan tireler → boşluk
    text = re.sub(r"(?<!\w)[-–—]|[-–—](?!\w)", " ", text)

    text = re.sub(r"[^\w\s\-çğıöşü]", " ", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [371]:
def process_data(ds=ds):
    train = ds["train"]#.select(range(100))

    def process_row(row):
        text = row["text"]
        text = strip_wikipedia_sections(text)
        text = strip_wikipedia_tables(text)
        text = strip_wikipedia_templates(text)
        text = strip_html(text)
        text = clean_text(text)
        row["tokens"] = text.split()
        return row
        
    train = train.map(process_row, num_proc=8, load_from_cache_file=False)
    return train["text"], train["tokens"]
data,tokens=process_data()

Map (num_proc=8):   0%|          | 0/534988 [00:00<?, ? examples/s]

TimeoutError: 

In [298]:
step=1
data[2][step*250:step*250+250]

"\n\nYaşamı\n\nErken dönemler\n\nAilesi \nAslen Samsunlu bir aileden gelmektedir. 2 Şubat 1895 tarihli Sicill-i Ahvâl Defteri, No.: 57, sayfa 73'e göre babası, 26 Mart 1812 Giresun doğumlu Mevlevîzade Saadetlû Ali Rıza Efendi'dir. Yine Sicill-i Ahvâl Defteri"

In [375]:
print(strip_wikipedia_templates(strip_wiki_markup(ds["train"]["text"][4067])))

Kobe Bean Bryant (23 Ağustos 1978; Philadelphia, Pensilvanya - 26 Ocak 2020; Calabasas, Kaliforniya), lakabı Black Mamba, NBA takımlarından Los Angeles Lakers'ın formasını giymiş Amerikalı profesyonel basketbolcudur. 1.98 boyunda olan Bryant şutör gard ve kısa forvet pozisyonunda görev almıştır. NBA tarihinin en önemli basketbolcularından birisi olarak gösterilmektedir.

Bryant liseden mezun olduktan sonra üniversiteye gitmeden direkt NBA'ye, Lakers'a adımını atmış ve tüm kariyerini bu kulüpte sürdürmüştü. Lise yıllarında Wilt Chamberlain'in 4 yıllık toplam sayı rekorunu kıran Kobe 1996 NBA Seçmeleri'nde Charlotte Hornets tarafından 13. sıradan draft edilse de sonra Lakers'a takas edilmiştir. 24 yaşındayken 3 NBA şampiyonluğu bulunan Kobe; Shaquille O'Neal ile kazandığı 3 şampiyonluktan tam 7 yıl sonra 2008-2009 sezonunda 4. şampiyonluğuna ulaştı ve kariyerinde birçok başarılar olmasına rağmen daha önce elde edemediği NBA finalleri en değerli oyuncusu da seçilmiş oldu. Ayrıca, Kobe 200

In [368]:
for idx, row in enumerate(ds["train"]):
    raw_id = row["id"]
    url = row["url"]
    title = row["title"]
    text = row["text"]
    
    # print(url) # İstersen burayı açabilirsin
    
    if "align" in clean_text(strip_html(strip_wikipedia_templates(strip_wikipedia_tables(text)))).split() and idx not in [4050, 79184, 113484, 219492]:
        print(f'find at: {raw_id} (idx: {idx})\n{text}')
        break # Sadece ilk bulduğunda durmasını istiyorsan break'i buraya koymalısın

find at: 13786 (idx: 4067)
Kobe Bean Bryant (23 Ağustos 1978; Philadelphia, Pensilvanya - 26 Ocak 2020; Calabasas, Kaliforniya), lakabı Black Mamba, NBA takımlarından Los Angeles Lakers'ın formasını giymiş Amerikalı profesyonel basketbolcudur. 1.98 boyunda olan Bryant şutör gard ve kısa forvet pozisyonunda görev almıştır. NBA tarihinin en önemli basketbolcularından birisi olarak gösterilmektedir.

Bryant liseden mezun olduktan sonra üniversiteye gitmeden direkt NBA'ye, Lakers'a adımını atmış ve tüm kariyerini bu kulüpte sürdürmüştü. Lise yıllarında Wilt Chamberlain'in 4 yıllık toplam sayı rekorunu kıran Kobe 1996 NBA Seçmeleri'nde Charlotte Hornets tarafından 13. sıradan draft edilse de sonra Lakers'a takas edilmiştir. 24 yaşındayken 3 NBA şampiyonluğu bulunan Kobe; Shaquille O'Neal ile kazandığı 3 şampiyonluktan tam 7 yıl sonra 2008-2009 sezonunda 4. şampiyonluğuna ulaştı ve kariyerinde birçok başarılar olmasına rağmen daha önce elde edemediği NBA finalleri en değerli oyuncusu da seçi

In [340]:
ds["train"]

Dataset({
    features: ['id', 'url', 'title', 'text'],
    num_rows: 534988
})

In [299]:
'|'.join(tokens[2][step*33:step*33+33])

'merkez|komitesi|başkanı|yaşamı|erken|dönemler|ailesi|aslen|samsunlu|bir|aileden|gelmektedir|2|şubat|1895|tarihli|sicill-i|ahvâl|defteri|no|57|sayfa|73e|göre|babası|26|mart|1812|giresun|doğumlu|mevlevîzade|saadetlû|ali'

In [360]:
full_counts = Counter()

for sentence in tokens:
    full_counts.update(sentence)

In [361]:
full_counts

Counter({'ve': 2861481,
         'bir': 1798786,
         'olarak': 715977,
         'bu': 701446,
         'ile': 620242,
         'için': 512807,
         'olan': 401436,
         'da': 387331,
         'de': 362921,
         '1': 357884,
         'tarafından': 348778,
         'align': 345061,
         'yılında': 334766,
         'sonra': 310697,
         'en': 305616,
         'daha': 305249,
         'ilk': 291465,
         '2': 251986,
         'the': 230223,
         'center': 207205,
         '3': 206342,
         'bağlı': 191573,
         'yer': 190662,
         'gibi': 189801,
         'büyük': 188749,
         'ise': 185672,
         'çok': 183909,
         'veya': 180915,
         '4': 175598,
         'arasında': 172213,
         'iki': 171313,
         'kadar': 168404,
         '5': 159309,
         'colspan': 150656,
         'left': 145973,
         'ayrıca': 145658,
         'd': 143375,
         'ancak': 142508,
         'oldu': 141729,
         'dış': 138981,
       